# Diffusion Posterior Sampling

A small deblurring walkthrough using a pretrained diffusion prior. See the README for the paper, course-template credits, and the archived example. Install dependencies separately and provide a local checkpoint and image.


In [ ]:
from pathlib import Path

import torch
from IPython.display import display
from PIL import Image

from dps.diffusion import create_diffusion
from dps.images import load_image, save_image
from dps.model import load_model
from dps.operators import GaussianBlur
from dps.sampling import sample_posterior


## Configuration

Measurement noise uses the normalized [-1, 1] image scale. Smaller step counts respace the diffusion schedule.


In [ ]:
checkpoint = Path("checkpoints/ffhq_10m.pt")
image_path = Path("data/00002.png")
output = Path("runs/notebook")
seed = 0
sigma_y = 0.05
guidance_scale = 0.3
steps = 1000
device = "cuda" if torch.cuda.is_available() else "cpu"


## Diffusion prior and measurement


In [ ]:
torch.manual_seed(seed)
model = load_model(checkpoint, device)
diffusion = create_diffusion(steps)
operator = GaussianBlur().to(device)
ground_truth = load_image(image_path, device)
with torch.no_grad():
    measurement = operator(ground_truth) + sigma_y * torch.randn_like(ground_truth)


## Posterior reconstruction

Guidance differentiates the measurement residual through the denoiser at each reverse step.


In [ ]:
estimate = sample_posterior(
    model, diffusion, operator, measurement, guidance_scale=guidance_scale,
)


## Save and view


In [ ]:
import json
from dps.__main__ import sha256

output.mkdir(parents=True, exist_ok=False)
for name, tensor in (("ground-truth", ground_truth), ("observation", measurement),
                     ("estimate", estimate)):
    path = output / f"{name}.png"
    save_image(tensor, path)
    print(name)
    display(Image.open(path))
settings = {"seed": seed, "sigma_y": sigma_y, "guidance_scale": guidance_scale,
            "steps": steps, "base_steps": 1000, "device": device,
            "blur_kernel_size": 61, "blur_sigma": 4.0,
            "checkpoint_sha256": sha256(checkpoint),
            "image_sha256": sha256(image_path), "torch_version": torch.__version__,
            "guidance_objective": "per-image residual L2 norm",
            "reverse_variance": "learned_range", "clip_denoised": True}
(output / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")
